In [0]:

storage_account_name = "adlsprimeabdel"
#CLE_D_ACCES par la clé copiée sur Azure
storage_account_key = "LA_VRAIE_KEY_1"

# Configuration de la connexion
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)
# Test de lecture du dossier Bronze

display(dbutils.fs.ls(f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"))

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#**Explorons pour comprendre la data**

In [0]:
df_silver = spark.read.format("csv")\
                  .option("header", True)\
                  .option("inferSchema", True)\
                  .load(f"abfss://bronze@adlsprimeabdel.dfs.core.windows.net/amazon_prime_titles.csv")

df_silver.display()

In [0]:
df_silver.printSchema()

# **DATA CLEANING**

In [0]:
df_silver.display()

In [0]:
df_silver = df_silver.na.fill({'rating': 'Unrated', 'country': 'Unknown'})
df_silver.display()

In [0]:
df_silver =df_silver.dropDuplicates()
df_silver.display()

In [0]:
df_silver = df_silver.na.fill({
    'rating': 'Unrated',
    'country': 'Unknown',
    'date_added': '01/01/2025',
    'release_year': '2020',
    'duration': 'Unknown',
    'listed_in': 'Unknown',
    'description': 'Unknown'
})

df_silver.display()

In [0]:
df_silver = df_silver.dropna('any')
df_silver.display()

In [0]:
# On renomme 'title' en 'content_title'
df_silver = df_silver.withColumnRenamed('title', 'content_title')

# On affiche le résultat
display(df_silver)

#**DATA TRANSFORMATION**

In [0]:
df_silver = df_silver.withColumn('is_country', when(col('country')=='Unknown', 0).otherwise(1))
df_silver.display()

In [0]:
# 1. On donne les droits d'accès
spark.conf.set("fs.azure.account.key.adlsprimeabdel.dfs.core.windows.net", "LA_VRAIE_KEY_1")

# 2. On écrit les données
df_silver.write.format("delta") \
  .mode("overwrite") \
  .save("abfss://silver@adlsprimeabdel.dfs.core.windows.net/amazon_prime_titles_silver")